# A2 — Listening and Accepting Connections: Observations

Server-side code lives in `../kvstore/server.py`. This notebook is for connecting clients and observing what happens.

**BUILD (from curriculum):** Extend A1 with `listen(backlog=5)` and `accept()`. Print the client address when a connection arrives. Test by connecting from a separate process.

**Workflow:** start your server in a terminal (`python kvstore/server.py`), then run cells here to connect to it.

## 1. Single client connection

Expected: the server prints the client's address (something like `('127.0.0.1', 54321)`). Note the client port is ephemeral — the kernel picked it for you because you didn't `bind()` on the client side.

In [85]:
import socket

PORT, HOSTNAME = 5001, "localhost"
import socket

def socketClient(hostname: str, port: int):
  tcp_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
  tcp_socket.connect((hostname, port))
  print("Connected to socket: ", tcp_socket.getsockname())

socketClient(HOSTNAME, PORT)

Connected to socket:  ('127.0.0.1', 61078)


## 2. Rapid multiple clients

Open 3 client connections in quick succession. The server, looping on `accept()`, dequeues them one at a time. The kernel holds any that arrive while the server is busy elsewhere — that's the **backlog queue**.

Expected: all 3 connect successfully and the server logs three addresses. Note the client ports — each is different (different ephemeral port per process/socket).

In [76]:
for i in range(3):
  socketClient(HOSTNAME, PORT)
  

Connected to socket:  ('127.0.0.1', 55634)
Connected to socket:  ('127.0.0.1', 55635)
Connected to socket:  ('127.0.0.1', 55636)


## 3. Overflow the backlog (optional)

To see what happens when the backlog fills up, modify your server to *not* call `accept()` (or sleep before calling it). Then try to open more than `backlog` connections from here.

Expected behavior is OS-dependent: on macOS/Linux, additional connections may be silently dropped or refused once the queue is full. The exact symptom varies. Note what you actually observe.

In [73]:
import socket, time

clients = []

for i in range(5):
  try:
    s = socketClient(HOSTNAME, PORT)
    clients.append(s)
  except (socket.timeout, ConnectionRefusedError) as e:
    print(f"client {i}: failed - {type(e).__name__}")
  time.sleep(0.1)


Connected to socket:  ('127.0.0.1', 55598)
Connected to socket:  ('127.0.0.1', 55599)
client 2: failed - TimeoutError
client 3: failed - TimeoutError
client 4: failed - TimeoutError
